<a href="https://colab.research.google.com/github/takatakamanbou/ML/blob/2025/ML2025_ex12notebookC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML ex12notebookC

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/ML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?ML)


---
## 主成分分析に関する実習
---

In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

---
### 計算問題

ほぼ線形代数の復習...

#### 問1

$\pmb{w}_1 = (1, 0, 1), \pmb{x}_1 = (1, 2, 3)$ のとき，次の値を求めなさい．
1. $\pmb{w}_1\cdot\pmb{x}_1$
1. $\Vert\pmb{w}_1\Vert$ および $\Vert\pmb{x}_1\Vert$

次のセルを実行すると，答えを確認できます（数値的に計算した結果なので誤差を含みますが）．

以下では， $\Vert\pmb{x}_1\Vert^2 = \pmb{x}_1 \cdot \pmb{x}_1 $ という性質を利用して $\Vert\pmb{x}_1\Vert$ を計算してます．

In [ ]:
w1 = np.array([1.0, 0.0, 1.0])
x1 = np.array([1.0, 2.0, 3.0])
print(f'w1 = {w1}')
print(f'x1 = {x1}')
print(f'w1 と x1 の内積 = {w1 @ x1}')
print(f'|w1| = {np.sqrt(w1 @ w1):.3f}')
print(f'|x1| = {np.sqrt(x1 @ x1):.3f}')

#### 問2

$\pmb{w}_2 = (0, 1, -1), \pmb{x}_2 = (4, 5, 6)$ のとき，次の値を求めなさい．$\pmb{w}_1, \pmb{x}_1$ は問1のものを使ってください．以下，前の問題で出てきた記号を説明せずに使ってる場合があります．
1. $\pmb{w}_1\cdot\pmb{x}_2$
1. $\pmb{w}_2\cdot\pmb{x}_1$
1. $\pmb{w}_2\cdot\pmb{x}_2$


In [ ]:
w2 = np.array([0.0, 1.0, -1.0])
x2 = np.array([4.0, 5.0, 6.0])
print(f'w2 = {w2}')
print(f'x2 = {x2}')
print(f'w1 と x2 の内積 = {w1 @ x2}')
print(f'w2 と x1 の内積 = {w2 @ x1}')
print(f'w2 と x2 の内積 = {w2 @ x2}')

#### 問3

次のセルを実行すると表示されるような，3行4列の行列を $X$ とします．
この行列を列ごとに眺めると，第1列は $\pmb{x}_1$ と等しく，第2列は $\pmb{x}_2$ と等しいことに注意しましょう．

In [ ]:
X = np.arange(1, 13).reshape((4, 3)).T
X

また，次のような 2行3列の行列を $W$ とします．こちらは，行ごとに眺めると，第1行が $\pmb{w}_1$ と等しく，第2行が $\pmb{w}_2$ と等しくなっています．

In [ ]:
W = np.vstack((w1, w2))
W

このとき，$Y = WX$ を手計算で求めなさい．

次のセルを実行すると，答えを確認できます．

In [ ]:
Y = W @ X
Y

結果を見ると，$Y$ の1列目には，$\pmb{w}_1\cdot\pmb{x}_1$ と $\pmb{w}_2\cdot\pmb{x}_1$ の値が入っており，2列目には，$\pmb{w}_1\cdot\pmb{x}_2$ と $\pmb{w}_2\cdot\pmb{x}_2$ の値が入っていることが分かります．他の列についても，同様のことが成り立っています．

この例では，$X$ は $3\times 4$ で，3次元ベクトルを4つならべたものと考えられます．このとき，$Y$ は $2\times 4$ の行列となっており，その各列は，$X$の1つの列を作っている 3 次元ベクトルと2つの3次元ベクトル $\pmb{w}_1, \pmb{w}_2$ との内積の値を表す2次元ベクトルとなっています．
つまり，$Y$ は，行列 $X$ が表す3次元のデータ4つを，2つの3次元ベクトル $\pmb{w}_1, \pmb{w}_2$ を使って2次元に次元削減したものとみなせます．

上記の例では，$X$, $Y$ ともに，データが列ベクトルの形で表されています．線形代数の教科書等で線形変換（一次変換）の話が出てくるときはほとんどこの形でしょう．しかし，コンピュータでデータを扱う場合は，一つのデータが1行で表される形の方が便利です．そういう場合は，行列を転置して考えれば ok です．

In [ ]:
Xt = X.T
Xt

In [ ]:
Yt = Xt @ W.T
Yt

---
### 数物情3次元データの主成分分析

「数学」「物理」「情報」の点数のデータに主成分分析を適用してみましょう（notebookBで使ったのと同じデータです）．

In [ ]:
# 数物情データを入手
! wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/PIP/mpi100-mac.csv
dfMPI = pd.read_csv('mpi100-mac.csv', index_col=0)
datMPI = dfMPI.to_numpy().astype(float)
dfMPI

平均を引いたデータを作ります．

In [ ]:
# 平均
Xmpi_mean = np.mean(dfMPI.to_numpy(), axis=0)
print('平均:', Xmpi_mean)

# 平均を引いて NumPy 配列にする
Xmpi = dfMPI.to_numpy() - Xmpi_mean
Nmpi, Dmpi = Xmpi.shape
print('Xmpi.shape:', Xmpi.shape) # Xmpi は 100 x 3
print(Xmpi[:5, :]) # 最初の5人分を表示

以下のセルを実行すると，`Xmpi` の分散共分散行列の固有値と固有ベクトルが求まります．
ここでは，特異値分解という手法を使ってそれらを求めていますが，説明は省きます．

In [ ]:
# 分散共分散行列の固有値と固有ベクトルを求める（Xの特異値分解経由で）
_, sval, Vt = np.linalg.svd(Xmpi, full_matrices=False)
eval = sval**2/Nmpi
U = Vt
for d in range(Dmpi):
    print(f'{d+1}番目の固有値:{eval[d]:.2f}   固有ベクトル:{U[d, :]}')

得られた3つの固有ベクトルのうち，対応する固有値の大きい2つ（1番目と2番目）を使って行列 `W` を構成し，これを用いて3次元の `Xmpi` を 2次元の `Y` に変換します．

In [ ]:
W = U[:2, :]
print(W, end=' ')
print(W.shape)
print()
Y = Xmpi @ W.T # y = Wx の計算
print(Y[:5, :], end=' ') # 最初の5人分を表示
print(Y.shape)

`Y` の散布図を描いてみましょう．

In [ ]:
nList = [44, 47, 41, 6] # 特徴的な4人のインデックス

fig, ax = plt.subplots(facecolor="white", figsize=(6, 6))
ax.scatter(Y[:, 0], Y[:, 1])
ax.scatter(Y[nList, 0], Y[nList, 1])
ax.axvline(0, linestyle='-', color='gray')
ax.axhline(0, linestyle='-', color='gray')
ax.set_xlim(-40, 40)
ax.set_ylim(-40, 40)
ax.set_aspect('equal')
ax.set_xlabel('$y_1$')
ax.set_ylabel('$y_2$')
#ax.legend()
for n in nList:
    plt.annotate(f'{n}', (Y[n, 0]+2, Y[n, 1]+2))
plt.show()

図にオレンジ色の点として描かれている4人について，元の点数，そこから平均を引いた値，それを変換して得られる値を表示すると次のようになります．

In [ ]:
nList = [44, 47, 41, 6]
for n in nList:
    print(f'{n:2d} {Xmpi[n]+Xmpi_mean} {Xmpi[n]} {Y[n]}')

固有値最大の固有ベクトルに対応する $\pmb{w}_1$ は

In [ ]:
W[0, :]

ですので， 3科目の点数からそれぞれの平均を引いた値を並べたベクトル $\pmb{x} = (\mbox{数学}, \mbox{物理}, \mbox{情報})$ に対して， $y_1 = \pmb{w}_1\cdot \pmb{x}$ は
おおよそ次のような式となっています．

$$
y_1 = 0.5 (\mbox{数学}) - 0.5(\mbox{物理}) - 0.7(\mbox{情報})
$$

したがって，「数学が高得点で他の2科目が低得点」なひとは $y_1$ が正となり，「数学が低得点で他の2科目が高得点」なひとは $y_1$ が負となります．上記の4人の点数でそのことを確認してみましょう．

一方，2番目のベクトル $\pmb{w}_2$ は

In [ ]:
W[1, :]

ですので，$y_2 = \pmb{w}_2\cdot\pmb{x}$ はおおよそ次のような式となっています．

$$
y_2 = 0.3 (\mbox{数学}) + 0.9(\mbox{物理}) - 0.4(\mbox{情報})
$$

「情報が低得点で他の2科目が高得点」なひとは  $y_2$  が正となり，「情報が高得点で他の2科目が低得点」なひとは  $y_2$  が負となります．こちらも上記の4人の点数でそのことを確認してみましょう．

---
### 外食への年間支出金額データの主成分分析

独立行政法人統計センターが作成している [教育用標準データセット(SSDSE)](https://www.nstac.go.jp/use/literacy/ssdse/) の中の「SSDSE-家計消費」というデータを使って，主成分分析の実験をやってみましょう．



#### データの入手と前処理

データセットを読み込んで前処理を行うコードを実行します． データについての説明も表示します．



In [ ]:
!wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/MVA/DiningOutSpendingData.py
from DiningOutSpendingData import DiningOutSpendingData
dosd = DiningOutSpendingData(dropcolumns=True)
kinki5 = dosd.kinki5()
tohoku6 = dosd.tohoku6()
dosd.info()

次のコードセルを実行すると，読み込んだデータに前処理を適用して，最初の5件のデータを表示します．9個の特徴量があります．

In [ ]:
# DataFrame 表示時の小数部の表示桁数
pd.options.display.precision = 1
# DataFrame を入手して最初の5件を表示
dfFeature = dosd.getDataFrame()
dfFeature.head(5)

上記のデータから平均を差し引いて，平均を 0 にしたデータ行列 `X` を作ります．データの次元数 D = 13, データ数 N = 47 です．

In [ ]:
# 平均を 0 にしたデータを X という名の NumPy array に
X = dosd.getArray()
Xm = np.mean(X, axis=0)
X -= Xm
print(X[:5, :], end=' ') # 最初の5件を表示
print(X.shape)

---
#### 主成分分析して可視化してみよう



数物情3次元データと同様に自前で主成分分析の計算を行ってもよいのですが，ここでは，Python の機械学習ライブラリである scikit-learn の主成分分析のクラス [sklearn.decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) を使ってみます．

In [ ]:
# PCAを適用
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca.fit(X)

# データ分散共分散行列の固有ベクトルのうち，固有値の大きい方から2つを取り出す
U = pca.components_[:2, :]

# 2つの固有ベクトルの値を表示
df_hoge = pd.DataFrame(index=dfFeature.columns[2:])
df_hoge['u1'] = U[0]
df_hoge['u2'] = U[1]
# 浮動小数点の表示形式を変更
pd.options.display.float_format = '{:.3f}'.format
df_hoge.T

データの分散共分散行列の最大固有値に対応する固有ベクトル（u1）は，全ての項目の値が正です．したがって，第1主成分スコア（u1 を用いてデータを変換して得られる値） y1 は，外食全体への支出が多い県で大きくなります．特に，和食，洋食，すし（外食）等の影響が大きいようです．

2番目に大きい固有値に対応する固有ベクトル（u2）の要素の値のうち，絶対値が大きいものを列挙すると，次のようになっています：
- 正: 中華そば，すし（外食），洋食，日本そば・うどん
- 負: 和食，喫茶代，ハンバーガー

したがって，「正」の項目に平均より多く支出し，「負」の項目に平均より少なく支出する県は，y2 の値が大きくなり，逆の傾向にある県は小さくなります．


これら2つの固有ベクトルを使って D = 9 次元のデータを 2 次元に変換してみましょう．

In [ ]:
# 2次元へ次元削減
Y = pca.fit_transform(X)
print('Y の最初の5行')
print(Y[:5, :], Y.shape)

数値を見てもわかりにくいので，散布図にしてみましょう．

In [ ]:
# グラフを描く
fig, ax = plt.subplots(facecolor="white", figsize=(8, 8))
ax.scatter(Y[:, 0], Y[:, 1], s=8)
ax.scatter(Y[kinki5, 0], Y[kinki5, 1], label='kinki5')
ax.scatter(Y[tohoku6, 0], Y[tohoku6, 1], label='tohoku')
ax.axvline(0, linestyle='-', color='gray')
ax.axhline(0, linestyle='-', color='gray')
ax.set_xlim(-5000, 9000)
ax.set_ylim(-7000, 7000)
ax.set_aspect('equal')
ax.legend()
for n in range(len(Y)):
    plt.annotate(f'{n}', (Y[n, 0]+20, Y[n, 1]+20))
plt.show()

#### ★★★ やってみよう ★★★

上記の結果を見て，次のことを考えよう．結果を紙媒体にメモしておこう．

- 東北6県は
    - 外食に使う金額が多い方？それとも少ない方？
    - 「中華そば，すし（外食），洋食，日本そば・うどん」と「和食，喫茶代，ハンバーガー」では，どちらの方に平均より多く支出する傾向がありそうか？
- 近畿5府県はどんな傾向？
